# MetroFlow: Exploratory Data Analysis and Diagnostics

This notebook analyses the prepared hourly ridership series for Times Sq-42 St / Port Authority Bus Terminal (station complex 611). Run `python -m metroflow.data.prepare` before running this notebook. All validation and splitting later in the project remains chronological.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

sns.set_theme(style='whitegrid', context='notebook')
DATA_PATH = Path('../data/processed/mta_hourly_station_611_clean.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp']).sort_values('timestamp')
df = df.set_index('timestamp').asfreq('h')
assert df['ridership'].notna().all(), 'Prepared series must not contain missing ridership values.'
df.head()

## Data profile and quality confirmation

The source is aggregated from fare/payment records to one observation per station complex and hour. `was_imputed=True` identifies the spring daylight-saving skipped hour, so it is auditable and can be excluded in sensitivity checks.

In [ ]:
profile = pd.Series({
    'records': len(df),
    'start': df.index.min(),
    'end': df.index.max(),
    'missing_ridership': int(df['ridership'].isna().sum()),
    'imputed_hours': int(df['was_imputed'].sum()),
    'minimum_hourly_entries': df['ridership'].min(),
    'maximum_hourly_entries': df['ridership'].max(),
    'mean_hourly_entries': df['ridership'].mean(),
})
profile

## Trend and seasonal structure

In [ ]:
daily = df['ridership'].resample('D').sum()
fig, ax = plt.subplots(figsize=(14, 4))
daily.plot(ax=ax, color='#1769aa', linewidth=0.7, label='Daily entries')
daily.rolling(28, center=True).mean().plot(ax=ax, color='#e65100', linewidth=2, label='28-day mean')
ax.set(title='Daily subway entries: Times Sq-42 St / Port Authority', xlabel='', ylabel='Entries')
ax.legend()
plt.show()

In [ ]:
hour_day = df.pivot_table(index='day_of_week', columns='hour', values='ridership', aggfunc='mean')
fig, ax = plt.subplots(figsize=(15, 4))
sns.heatmap(hour_day, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Mean hourly entries'})
ax.set(title='Typical hourly footfall by day of week', xlabel='Hour of day', ylabel='Day of week (0 = Monday)')
plt.show()

## Decomposition and stationarity

STL is applied to daily totals with a seven-day period to separate weekly seasonality from trend. The Augmented Dickey-Fuller test is reported on the hourly target and on its weekly difference. Model selection will use the differenced form only if diagnostics and validation support it.

In [ ]:
stl = STL(daily, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(14, 8)
plt.show()

def adf_summary(series, label):
    statistic, p_value, lags, observations, *_ = adfuller(series.dropna(), autolag='AIC')
    return {'series': label, 'adf_statistic': statistic, 'p_value': p_value, 'lags_used': lags, 'observations': observations}

pd.DataFrame([
    adf_summary(df['ridership'], 'hourly ridership'),
    adf_summary(df['ridership'].diff(24 * 7), 'weekly-differenced ridership'),
])

In [ ]:
# Plot a recent window so daily and weekly lags are readable.
recent = df['ridership'].tail(24 * 56)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(recent, lags=24 * 14, ax=axes[0], zero=False)
plot_pacf(recent, lags=72, ax=axes[1], method='ywm')
axes[0].set_title('ACF: recent hourly ridership')
axes[1].set_title('PACF: recent hourly ridership')
plt.tight_layout()
plt.show()

## Outliers and modelling implications

The IQR screen identifies candidate unusual hours for investigation. These observations are retained because special events and disruptions are part of the operational problem. The model stage will compare seasonal-naive, ETS, and SARIMAX using expanding-window validation and MAE, RMSE, and WAPE.

In [ ]:
q1, q3 = df['ridership'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
candidates = df[(df['ridership'] < lower) | (df['ridership'] > upper)]
print(f'IQR candidate unusual hours: {len(candidates):,} ({len(candidates)/len(df):.1%})')
candidates[['ridership', 'day_of_week', 'hour', 'was_imputed']].sort_values('ridership', ascending=False).head(10)

## EDA conclusion template

After execution, record the observed daily/weekly seasonal peaks, the trend pattern, ADF interpretation, and whether the ACF supports daily and weekly lags. These evidence-based observations will be used to justify the candidate models in `02_model_experiments.ipynb`; do not claim a model is best before chronological validation.